In [1]:
import importlib
import weights_cuda
import spike_engine_cuda
importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)
from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus
import cupy as cp


In [17]:
# N = 50, lifetime = 100000 # can it finish in a minute?
N = 256
lifetime = 10000
engine = SpikeEngineCUDA(
    square_torus(N),
    (N, N),
    use_k2tree=True,
    verify_k2tree=False,
    verify_progress_every=2000,
    decay_rate=0.95,
    rank=64
)


Constructing weight matrix...
Weights constructed.


In [18]:
# Estimate bifurcation threshold and set constant weights near it
w_accum, w_instant = engine.estimate_bifurcation_weight(input_period=1)
target, _, _ = engine.set_constant_weights_near_bifurcation(input_period=1, scale=1.5, freeze_learning=True)
print(f"w_accum={w_accum:.6f} w_instant={w_instant:.6f} target={target:.6f}")


w_accum=0.855000 w_instant=0.900000 target=1.282500


In [21]:
input_neuron = (N * N) // 2 + N//2
engine.set_input_neurons([input_neuron])
inputspikes = cp.ones((lifetime, 1))

engine.start_static_record(
    inputspikes,
    lifetime,
    "cuda_test_9.spire.gz",
    record_membrane=True,
    full_decay=True,
    compression_level=4,
    compression_async=True
)
# rsync -avP user@remote:/path/to/file /local/path


100%|████████████████████████████████████████████████████████████████████████| 10000/10000 [04:01<00:00, 41.45it/s]

Recording saved: cuda_test_9.spire.gz


In [ ]:
# Optional: validate neighbors for a random neuron
idx = 123
print("neighbors:", engine.weights.get_neighbors(idx))
